<a href="https://colab.research.google.com/github/EmePin/Analisis-de-datos/blob/main/Fake_News_2_0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os
!pip install tensorflow

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

import gradio as gr
import joblib


In [2]:
# Descargar dataset
path = kagglehub.dataset_download("subho117/fake-news-detection-using-machine-learning")

df = pd.read_csv(os.path.join(path, "News.csv"))

# Usaremos título (rápido y efectivo)
df = df[['title', 'class']].dropna()

X = df['title'].astype(str).values
y = df['class'].astype(int).values


Using Colab cache for faster access to the 'fake-news-detection-using-machine-learning' dataset.


In [3]:
VOCAB_SIZE = 5000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=VOCAB_SIZE)# convierte palabras → números
tokenizer.fit_on_texts(X) # Diccionario

X_seq = tokenizer.texts_to_sequences(X)
# Convierte cada noticia en una lista de números
# Ejemplo: "economy grows fast" → [45, 102, 78]
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN)
# Ajusta todas las noticias al mismo tamaño
# Si son cortas → agrega ceros (padding)
# Si son largas → las recorta


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)


Epoch 1/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 240s 520ms/step - accuracy: 0.9379 - loss: 0.1614 - val_accuracy: 0.9673 - val_loss: 0.0903
Epoch 2/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 256s 508ms/step - accuracy: 0.9779 - loss: 0.0615 - val_accuracy: 0.9709 - val_loss: 0.0850
Epoch 3/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 232s 515ms/step - accuracy: 0.9873 - loss: 0.0359 - val_accuracy: 0.9681 - val_loss: 0.1001
Epoch 4/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 228s 507ms/step - accuracy: 0.9933 - loss: 0.0219 - val_accuracy: 0.9672 - val_loss: 0.1109
Epoch 5/5
341/450 ━━━━━━━━━━━━━━━━━━━━ 52s 480ms/step - accuracy: 0.9966 - loss: 0.0114

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)


In [ ]:
model.save("lstm_fake_news.keras")


joblib.dump(tokenizer, "tokenizer_lstm.pkl")


In [12]:
def predict_news_lstm(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN)

    pred = model.predict(padded)[0][0]

    if pred > 0.5:
        label = "REAL"
        confidence = pred
    else:
        label = "FALSA"
        confidence = 1 - pred

    return f"{label} ({confidence*100:.1f}% de confianza)"

In [14]:
app = gr.Interface(
    fn=predict_news_lstm,

    inputs=gr.Textbox(
        label="Escribe una noticia en inglés",
        placeholder="Ej: Vaccines contain microchips..."
    ),

    outputs=gr.Text(label="Resultado"),

    title="📰 Detector de Fake News",
    description="Escribe una noticia en inglés y la IA te dirá si es REAL o FALSA"
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d6774173bf681830c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
